In [ ]:
import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy import stats
ignore = [".ipynb_checkpoints", "benchmark_data"]
data_dir = Path(r"D:\Download\factor_results")

In [6]:
factor_folder = [item for item in os.listdir(data_dir) if item not in ignore]
for factor in factor_folder:
    save_dir = data_dir / factor
    ic = pd.read_csv(
        save_dir / "ic.csv",
        index_col="date",
        parse_dates=["date"],
    ).iloc[:, 0].dropna()
    print(f"{factor}: {ic.mean()}")

CSAD 模型（基于“小型”板块）: 0.019646703673482587
CSAD 模型（基于时序回归变形）: 0.01427929514467051
CSAD 模型（基于板块中地位）: 0.031724281945735455
LSV 模型: 0.006367479691169617
SLSV 模型（对称 LSV 模型）: -0.0015701674920779144
前景价值（TK）: -0.003318387434368924
基于 PB 的前景价值 TK: -0.0013550991776773685
基于 PE 的分析师锚定效应偏差（CAF_EP）: 0.014255680179169242
基于市场波动率的注意力捕捉: 0.03062714566250019
基于异常收益的注意力捕捉: 0.026899155112612785
基于异常收益的注意力溢出（SPILL）: 0.04475829861278153
基于换手率的注意力捕捉: 0.029249844094572184
基于换手率的注意力溢出（SPILL）: 0.03781257631261429
基于沪深总成交额的注意力捕捉: 0.032400973214168564
基于涨跌停的注意力捕捉: 0.03192754020787116
平均异常日收益率（ABNRETAVG）: 0.06085570304219788
平均换手率（TURNAVG）: 0.043783984886482066
最大异常日收益率（ABNRETD）: 0.056574783644962316
未实现盈利量（CGO）: 0.03191226271986045
每日平均异常成交（ABNVOLAVG）: 0.04143068005286419
每日最大异常成交量（ABNVOLD）: 0.04358133401626941
衰减加权月频成交量（ATTN）: 0.010209980531909383


In [ ]:
TRADING_DAYS = 252
RISK_FREE_RATE = 0.0

# 修改为实际保存目录
save_dir = data_dir / "平均异常日收益率（ABNRETAVG）"

# ========= 1. 读取本地回测参数与 IC 数据 =========
with open(save_dir / "params.json", "r", encoding="utf-8") as f:
    params = json.load(f)

ic = pd.read_csv(
    save_dir / "ic.csv",
    index_col="date",
    parse_dates=["date"],
).iloc[:, 0].dropna()

ic.name = "rank_ic"

# ========= 2. IC 指标 =========
ic_t_stat, ic_p_value = stats.ttest_1samp(ic, 0)
ic_std = ic.std(ddof=1)

ic_metrics = pd.Series({
    "样本天数": len(ic),
    "平均 Rank IC": ic.mean(),
    "Rank IC 标准差": ic_std,
    "ICIR": ic.mean() / ic_std,
    "年化 ICIR": ic.mean() / ic_std * np.sqrt(TRADING_DAYS),
    "IC t 值": ic_t_stat,
    "IC p 值": ic_p_value,
    "正 IC 占比": (ic > 0).mean(),
    "负 IC 占比": (ic < 0).mean(),
    "IC 偏度": ic.skew(),
    "IC 峰度": ic.kurtosis(),
    "IC 一阶自相关": ic.autocorr(lag=1),
    "累计 Rank IC": ic.cumsum().iloc[-1],
})


def format_number(x):
    """普通数值保留 3 位小数。"""
    if pd.isna(x):
        return ""
    return f"{x:,.3f}"


ic_metrics_df = ic_metrics.to_frame("Result")

display(
    ic_metrics_df.style
    .format({"Result": format_number})
    .format(
        {"Result": "{:,.0f}"},
        subset=pd.IndexSlice[["样本天数"], ["Result"]],
    )
    .format(
        {"Result": "{:.3e}"},
        subset=pd.IndexSlice[["IC p 值"], ["Result"]],
    )
)

# ========= 3. 读取本地分组累计收益、换手率 =========
# CSV 原始格式：行=q0~q9；列=日期；值=累计收益率
quantile_factor_returns = pd.read_csv(
    save_dir / "quantile_factor_returns.csv",
    index_col=0,
).apply(pd.to_numeric, errors="coerce")

quantile_factor_returns.columns = pd.to_datetime(
    quantile_factor_returns.columns
)

quantile_turnover = pd.read_csv(
    save_dir / "quantile_turnover.csv",
    index_col=0,
).apply(pd.to_numeric, errors="coerce")

quantile_turnover.columns = pd.to_datetime(
    quantile_turnover.columns
)

# 转为：行=日期；列=q0~q9
group_cumret = quantile_factor_returns.T.sort_index()

# 累计收益率 -> 净值 -> 日收益率
group_nav = 1 + group_cumret

# 禁止 pandas 用前值填充缺失净值，避免把缺失日错误视为 0 收益
group_daily_ret = (
    group_nav
    .pct_change(fill_method=None)
    .dropna(how="all")
)

# 换手率转置并匹配收益率日期
group_turnover = (
    quantile_turnover.T
    .sort_index()
    .reindex(group_daily_ret.index)
)

# ========= 4. 从本地全量基准库中截取本次回测区间 =========
benchmark_file = data_dir / "benchmark_data" / "000985.XSHG_close.csv"

benchmark_close_all = pd.read_csv(
    benchmark_file,
    index_col="date",
    parse_dates=["date"],
).iloc[:, 0]

benchmark_close_all = (
    pd.to_numeric(benchmark_close_all, errors="coerce")
    .dropna()
    .sort_index()
)

# 先对全历史价格计算收益率，再按本次策略日期截取。
# 这样首日收益率也会正确使用前一个交易日的收盘价。
benchmark_ret_all = benchmark_close_all.pct_change()

backtest_start = group_daily_ret.index.min()
backtest_end = group_daily_ret.index.max()

benchmark_ret = (
    benchmark_ret_all
    .loc[backtest_start:backtest_end]
    .reindex(group_daily_ret.index)
)

print(
    "本次绩效计算使用的基准区间：",
    benchmark_ret.index.min().date(),
    "至",
    benchmark_ret.index.max().date(),
)

# 这样不会在基准缺失时静默跳过日期，导致策略年化收益与基准年化收益的样本期不同。
benchmark_ret = benchmark_ret.reindex(group_daily_ret.index)

missing_benchmark_days = benchmark_ret.isna()

if missing_benchmark_days.any():
    missing_dates = benchmark_ret.index[missing_benchmark_days].strftime(
        "%Y-%m-%d"
    ).tolist()

    raise ValueError(
        "基准数据未覆盖以下策略交易日，无法进行严格的相对绩效计算："
        f"{missing_dates[:10]}"
    )

print(
    "策略与基准共同样本天数：",
    len(group_daily_ret.index),
)

# ========= 5. 单组合绩效函数 =========
def performance_metrics(daily_ret, benchmark_ret=None, annual_rf=0.0):
    daily_ret = daily_ret.dropna()

    if benchmark_ret is not None:
        pair = pd.concat(
            [
                daily_ret.rename("strategy"),
                benchmark_ret.rename("benchmark"),
            ],
            axis=1,
        ).dropna()

        if len(pair) < 2:
            return pd.Series(dtype=float)

        # 只要计算基准相关指标，所有策略指标也基于共同日期。
        daily_ret = pair["strategy"]
        benchmark_ret = pair["benchmark"]

    if len(daily_ret) < 2:
        return pd.Series(dtype=float)

    rf_daily = (1 + annual_rf) ** (1 / TRADING_DAYS) - 1
    excess_ret = daily_ret - rf_daily

    nav = (1 + daily_ret).cumprod()
    cumulative_return = nav.iloc[-1] - 1
    annual_return = nav.iloc[-1] ** (
        TRADING_DAYS / len(daily_ret)
    ) - 1

    daily_std = daily_ret.std(ddof=1)
    annual_volatility = daily_std * np.sqrt(TRADING_DAYS)

    sharpe = (
        excess_ret.mean() / excess_ret.std(ddof=1)
        * np.sqrt(TRADING_DAYS)
        if excess_ret.std(ddof=1) > 0 else np.nan
    )

    downside_volatility = (
        np.sqrt(np.mean(np.minimum(excess_ret, 0) ** 2))
        * np.sqrt(TRADING_DAYS)
    )

    sortino = (
        excess_ret.mean() * TRADING_DAYS / downside_volatility
        if downside_volatility > 0 else np.nan
    )

    drawdown = nav / nav.cummax() - 1
    max_drawdown = drawdown.min()

    calmar = (
        annual_return / abs(max_drawdown)
        if max_drawdown < 0 else np.nan
    )

    output = {
        "累计收益率": cumulative_return,
        "年化收益率": annual_return,
        "年化波动率": annual_volatility,
        "夏普比率": sharpe,
        "Sortino 比率": sortino,
        "最大回撤": max_drawdown,
        "Calmar 比率": calmar,
        "日胜率": (daily_ret > 0).mean(),
        "平均日收益": daily_ret.mean(),
        "日收益偏度": daily_ret.skew(),
        "日收益峰度": daily_ret.kurtosis(),
    }

    if benchmark_ret is not None:
        r = daily_ret
        b = benchmark_ret

        benchmark_annual_return = (
            (1 + b).prod() ** (TRADING_DAYS / len(b)) - 1
        )

        beta = (
            np.cov(r, b, ddof=1)[0, 1]
            / np.var(b, ddof=1)
        )

        alpha_daily = (
            (r - rf_daily).mean()
            - beta * (b - rf_daily).mean()
        )

        active_ret = r - b
        tracking_error = (
            active_ret.std(ddof=1) * np.sqrt(TRADING_DAYS)
        )

        information_ratio = (
            active_ret.mean() / active_ret.std(ddof=1)
            * np.sqrt(TRADING_DAYS)
            if active_ret.std(ddof=1) > 0 else np.nan
        )

        # 几何超额收益：策略净值相对基准净值的年化增速
        relative_nav = ((1 + r) / (1 + b)).cumprod()
        annual_geometric_excess = (
            relative_nav.iloc[-1]
            ** (TRADING_DAYS / len(relative_nav))
            - 1
        )

        output.update({
            "基准年化收益率": benchmark_annual_return,
            "年化超额收益（几何）": annual_geometric_excess,
            "年化收益差（算术）": (
                annual_return - benchmark_annual_return
            ),
            "Alpha（年化）": alpha_daily * TRADING_DAYS,
            "Beta": beta,
            "跟踪误差": tracking_error,
            "信息比率": information_ratio,
        })

    return pd.Series(output)

# ========= 6. 十个分组、total 与多空组合 =========
group_metrics = pd.DataFrame({
    group: performance_metrics(
        group_daily_ret[group],
        benchmark_ret,
        annual_rf=RISK_FREE_RATE,
    )
    for group in group_daily_ret.columns
}).T

# total：十个分组每日等权组合
total_daily_ret = group_daily_ret.mean(axis=1)
total_nav = (1 + total_daily_ret).cumprod()
total_cumret = total_nav - 1

TOTAL_GROUP = "total（十组等权）"

# 🚨严格的“全股票池等权组合”无法从十个分组的累计收益曲线完全还原，因为各组有效股票数、分组边界与缺失股票可能不同。
# 若要计算它，需要额外保存每日分组明细或逐股票收益数据。
group_metrics.loc[TOTAL_GROUP] = performance_metrics(
    total_daily_ret,
    benchmark_ret,
    annual_rf=RISK_FREE_RATE,
)

# 注意：确认 q0 是低因子值组、q9 是高因子值组后再使用
LONG_GROUP = "q0"
SHORT_GROUP = "q9"

long_short_daily_ret = (
    group_daily_ret[LONG_GROUP]
    - group_daily_ret[SHORT_GROUP]
)

# 多空策略本身通常不再与宽基指数比较
group_metrics.loc[f"long_short_{LONG_GROUP}_{SHORT_GROUP}"] = (
    performance_metrics(
        long_short_daily_ret,
        benchmark_ret=benchmark_ret,
        annual_rf=RISK_FREE_RATE,
    )
)

display(group_metrics.style.format(format_number))

# ========= 7. 换手率汇总 =========
turnover_metrics = pd.DataFrame({
    "平均日换手率": group_turnover.mean(),
    "中位数日换手率": group_turnover.median(),
    "年化换手率": group_turnover.mean() * TRADING_DAYS,
    "最大单日换手率": group_turnover.max(),
})

turnover_metrics.loc[TOTAL_GROUP] = {
    "平均日换手率": group_turnover.mean(axis=1).mean(),
    "中位数日换手率": group_turnover.mean(axis=1).median(),
    "年化换手率": group_turnover.mean(axis=1).mean() * TRADING_DAYS,
    "最大单日换手率": group_turnover.mean(axis=1).max(),
}

# q0 多头 + q9 空头。
# 与 long_short_daily_ret = q0 - q9 的收益定义一致：
# 两侧各持有 100% 名义本金，因此总换手率 = 两腿换手率之和。
long_short_turnover = group_turnover[
    [LONG_GROUP, SHORT_GROUP]
].sum(axis=1, min_count=2)

turnover_metrics.loc[
    f"long_short_{LONG_GROUP}_{SHORT_GROUP}"
] = {
    "平均日换手率": long_short_turnover.mean(),
    "中位数日换手率": long_short_turnover.median(),
    "年化换手率": long_short_turnover.mean() * TRADING_DAYS,
    "最大单日换手率": long_short_turnover.max(),
}

display(turnover_metrics.style.format(format_number))

,Result
样本天数,"1,454"
平均 Rank IC,0.061
Rank IC 标准差,0.137
ICIR,0.444
年化 ICIR,7.041
IC t 值,16.913
IC p 值,1.032e-58
正 IC 占比,0.668
负 IC 占比,0.332
IC 偏度,-0.114


本次绩效计算使用的基准区间： 2020-01-06 至 2025-12-31
策略与基准共同样本天数： 1453


,累计收益率,年化收益率,年化波动率,夏普比率,Sortino 比率,最大回撤,Calmar 比率,日胜率,平均日收益,日收益偏度,日收益峰度,基准年化收益率,年化超额收益（几何）,年化收益差（算术）,Alpha（年化）,Beta,跟踪误差,信息比率
q0,2.695,0.254,0.224,1.125,1.551,-0.334,0.762,0.574,0.001,-0.828,10.270,0.046,0.200,0.209,0.192,0.931,0.125,1.497
q1,2.735,0.257,0.229,1.114,1.536,-0.312,0.823,0.570,0.001,-0.855,10.240,0.046,0.202,0.211,0.193,0.963,0.124,1.541
q2,2.524,0.244,0.232,1.060,1.465,-0.308,0.793,0.569,0.001,-0.766,10.197,0.046,0.190,0.199,0.181,0.994,0.118,1.530
q3,2.144,0.220,0.234,0.966,1.337,-0.307,0.715,0.562,0.001,-0.707,9.937,0.046,0.167,0.174,0.160,1.026,0.113,1.434
q4,1.887,0.202,0.237,0.897,1.238,-0.305,0.662,0.553,0.001,-0.717,9.498,0.046,0.149,0.156,0.144,1.057,0.106,1.391
q5,1.616,0.182,0.239,0.818,1.126,-0.301,0.604,0.554,0.001,-0.669,8.714,0.046,0.130,0.136,0.126,1.083,0.102,1.282
q6,1.323,0.157,0.246,0.718,0.989,-0.319,0.493,0.546,0.001,-0.651,8.788,0.046,0.107,0.112,0.104,1.124,0.102,1.100
q7,0.902,0.118,0.253,0.568,0.778,-0.371,0.318,0.535,0.001,-0.590,6.689,0.046,0.069,0.072,0.069,1.158,0.106,0.743
q8,-0.428,-0.092,0.283,-0.199,-0.266,-0.643,-0.144,0.515,-0.000,-0.459,4.863,0.046,-0.132,-0.138,-0.136,1.226,0.148,-0.817
q9,-0.587,-0.142,0.297,-0.367,-0.489,-0.745,-0.191,0.511,-0.000,-0.392,4.567,0.046,-0.180,-0.188,-0.191,1.263,0.164,-1.062


,平均日换手率,中位数日换手率,年化换手率,最大单日换手率
q0,0.101,0.096,25.558,0.375
q1,0.216,0.207,54.449,0.735
q2,0.253,0.240,63.659,0.837
q3,0.266,0.253,66.953,0.897
q4,0.266,0.253,67.134,0.872
q5,0.257,0.244,64.760,0.864
q6,0.239,0.229,60.275,0.822
q7,0.213,0.204,53.585,0.773
q8,0.133,0.131,33.445,0.550
q9,0.071,0.068,17.980,0.312


---

### 保存summary到本地

In [15]:
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
from scipy import stats

TRADING_DAYS = 252
RISK_FREE_RATE = 0.0

data_dir = Path(r"D:\Download\factor_results")
ignore_folder_names = {".ipynb_checkpoints", "benchmark_data"}


def format_value(value, scientific=False, integer=False):
    """文本输出格式：普通数值三位小数，p 值科学计数法。"""
    if pd.isna(value):
        return ""

    if integer:
        return f"{int(value):,}"

    if scientific:
        return f"{value:.3e}"

    return f"{value:,.3f}"


def dataframe_to_text(
    df,
    scientific_rows=None,
    integer_rows=None,
):
    """将 DataFrame 转为适合 summary.txt 的制表符文本。"""
    scientific_rows = set(scientific_rows or [])
    integer_rows = set(integer_rows or [])

    output = df.copy().astype(object)

    for row in output.index:
        for column in output.columns:
            output.loc[row, column] = format_value(
                output.loc[row, column],
                scientific=row in scientific_rows,
                integer=row in integer_rows,
            )

    output.index.name = ""
    return output.to_csv(
        sep="\t",
        lineterminator="\n",
    )


def performance_metrics(daily_ret, benchmark_ret=None, annual_rf=0.0):
    """根据日收益率计算组合绩效指标。"""
    daily_ret = daily_ret.dropna()

    if benchmark_ret is not None:
        pair = pd.concat(
            [
                daily_ret.rename("strategy"),
                benchmark_ret.rename("benchmark"),
            ],
            axis=1,
        ).dropna()

        if len(pair) < 2:
            return pd.Series(dtype=float)

        daily_ret = pair["strategy"]
        benchmark_ret = pair["benchmark"]

    if len(daily_ret) < 2:
        return pd.Series(dtype=float)

    rf_daily = (1 + annual_rf) ** (1 / TRADING_DAYS) - 1
    excess_ret = daily_ret - rf_daily

    nav = (1 + daily_ret).cumprod()

    cumulative_return = nav.iloc[-1] - 1
    annual_return = nav.iloc[-1] ** (
        TRADING_DAYS / len(daily_ret)
    ) - 1

    annual_volatility = (
        daily_ret.std(ddof=1) * np.sqrt(TRADING_DAYS)
    )

    sharpe = (
        excess_ret.mean() / excess_ret.std(ddof=1)
        * np.sqrt(TRADING_DAYS)
        if excess_ret.std(ddof=1) > 0 else np.nan
    )

    downside_volatility = (
        np.sqrt(np.mean(np.minimum(excess_ret, 0) ** 2))
        * np.sqrt(TRADING_DAYS)
    )

    sortino = (
        excess_ret.mean() * TRADING_DAYS / downside_volatility
        if downside_volatility > 0 else np.nan
    )

    drawdown = nav / nav.cummax() - 1
    max_drawdown = drawdown.min()

    calmar = (
        annual_return / abs(max_drawdown)
        if max_drawdown < 0 else np.nan
    )

    output = {
        "累计收益率": cumulative_return,
        "年化收益率": annual_return,
        "年化波动率": annual_volatility,
        "夏普比率": sharpe,
        "Sortino 比率": sortino,
        "最大回撤": max_drawdown,
        "Calmar 比率": calmar,
        "日胜率": (daily_ret > 0).mean(),
        "平均日收益": daily_ret.mean(),
        "日收益偏度": daily_ret.skew(),
        "日收益峰度": daily_ret.kurtosis(),
    }

    if benchmark_ret is not None:
        r = daily_ret
        b = benchmark_ret

        benchmark_annual_return = (
            (1 + b).prod() ** (TRADING_DAYS / len(b)) - 1
        )

        beta = (
            np.cov(r, b, ddof=1)[0, 1]
            / np.var(b, ddof=1)
        )

        alpha_daily = (
            (r - rf_daily).mean()
            - beta * (b - rf_daily).mean()
        )

        active_ret = r - b

        tracking_error = (
            active_ret.std(ddof=1) * np.sqrt(TRADING_DAYS)
        )

        information_ratio = (
            active_ret.mean() / active_ret.std(ddof=1)
            * np.sqrt(TRADING_DAYS)
            if active_ret.std(ddof=1) > 0 else np.nan
        )

        relative_nav = ((1 + r) / (1 + b)).cumprod()

        annual_geometric_excess = (
            relative_nav.iloc[-1]
            ** (TRADING_DAYS / len(relative_nav))
            - 1
        )

        output.update({
            "基准年化收益率": benchmark_annual_return,
            "年化超额收益（几何）": annual_geometric_excess,
            "年化收益差（算术）": (
                annual_return - benchmark_annual_return
            ),
            "Alpha（年化）": alpha_daily * TRADING_DAYS,
            "Beta": beta,
            "跟踪误差": tracking_error,
            "信息比率": information_ratio,
        })

    return pd.Series(output)


def group_sort_key(group_name):
    """确保 q0、q1、...、q9 按数值顺序排序。"""
    match = re.fullmatch(r"q(\d+)", str(group_name))

    if match:
        return (0, int(match.group(1)))

    return (1, str(group_name))


def build_summary(save_dir):
    """从单个因子目录的本地 CSV 生成 summary.txt。"""
    params_file = save_dir / "params.json"
    ic_file = save_dir / "ic.csv"
    return_file = save_dir / "quantile_factor_returns.csv"
    turnover_file = save_dir / "quantile_turnover.csv"

    required_files = [
        params_file,
        ic_file,
        return_file,
        turnover_file,
    ]

    missing_files = [
        file.name for file in required_files
        if not file.exists()
    ]

    if missing_files:
        raise FileNotFoundError(
            f"缺少文件：{', '.join(missing_files)}"
        )

    with open(params_file, "r", encoding="utf-8") as f:
        params = json.load(f)

    # ---------- IC 指标 ----------
    ic = pd.read_csv(
        ic_file,
        index_col="date",
        parse_dates=["date"],
    ).iloc[:, 0].dropna()

    ic_std = ic.std(ddof=1)
    ic_t_stat, ic_p_value = stats.ttest_1samp(ic, 0)

    ic_metrics = pd.Series({
        "样本天数": len(ic),
        "平均 Rank IC": ic.mean(),
        "Rank IC 标准差": ic_std,
        "ICIR": ic.mean() / ic_std if ic_std > 0 else np.nan,
        "年化 ICIR": (
            ic.mean() / ic_std * np.sqrt(TRADING_DAYS)
            if ic_std > 0 else np.nan
        ),
        "IC t 值": ic_t_stat,
        "IC p 值": ic_p_value,
        "正 IC 占比": (ic > 0).mean(),
        "负 IC 占比": (ic < 0).mean(),
        "IC 偏度": ic.skew(),
        "IC 峰度": ic.kurtosis(),
        "IC 一阶自相关": ic.autocorr(lag=1),
        "累计 Rank IC": ic.cumsum().iloc[-1],
    })

    # ---------- 分组累计收益 -> 日收益 ----------
    quantile_factor_returns = pd.read_csv(
        return_file,
        index_col=0,
    ).apply(pd.to_numeric, errors="coerce")

    quantile_factor_returns.columns = pd.to_datetime(
        quantile_factor_returns.columns
    )

    quantile_turnover = pd.read_csv(
        turnover_file,
        index_col=0,
    ).apply(pd.to_numeric, errors="coerce")

    quantile_turnover.columns = pd.to_datetime(
        quantile_turnover.columns
    )

    group_cumret = quantile_factor_returns.T.sort_index()
    group_nav = 1 + group_cumret

    group_daily_ret = (
        group_nav
        .pct_change(fill_method=None)
        .dropna(how="all")
    )

    group_turnover = (
        quantile_turnover.T
        .sort_index()
        .reindex(group_daily_ret.index)
    )

    # ---------- 本地基准 ----------
    benchmark = params.get("benchmark", "000985.XSHG")

    benchmark_file = (
        data_dir
        / "benchmark_data"
        / f"{benchmark}_close.csv"
    )

    if not benchmark_file.exists():
        raise FileNotFoundError(
            f"未找到基准文件：{benchmark_file}"
        )

    benchmark_close = pd.read_csv(
        benchmark_file,
        index_col="date",
        parse_dates=["date"],
    ).iloc[:, 0]

    benchmark_close = (
        pd.to_numeric(benchmark_close, errors="coerce")
        .dropna()
        .sort_index()
    )

    # 必须先用全历史价格计算收益率，再截取本次回测期
    benchmark_ret_all = benchmark_close.pct_change(fill_method=None)

    benchmark_ret = benchmark_ret_all.reindex(
        group_daily_ret.index
    )

    if benchmark_ret.isna().any():
        missing_dates = benchmark_ret.index[
            benchmark_ret.isna()
        ].strftime("%Y-%m-%d").tolist()

        raise ValueError(
            "基准未覆盖策略日期："
            f"{missing_dates[:10]}"
        )

    # ---------- 分组绩效 ----------
    group_names = sorted(
        group_daily_ret.columns,
        key=group_sort_key,
    )

    group_metrics = pd.DataFrame({
        group: performance_metrics(
            group_daily_ret[group],
            benchmark_ret=benchmark_ret,
            annual_rf=RISK_FREE_RATE,
        )
        for group in group_names
    }).T

    # total 是十个分组每日等权，不是严格的逐股票等权
    total_daily_ret = group_daily_ret[group_names].mean(axis=1)
    total_group_name = "total（十组等权）"

    group_metrics.loc[total_group_name] = performance_metrics(
        total_daily_ret,
        benchmark_ret=benchmark_ret,
        annual_rf=RISK_FREE_RATE,
    )

    # q0 - 最后一组，例如 q0 - q9 或 q0 - q4
    long_group = group_names[0]
    short_group = group_names[-1]
    long_short_name = f"long_short_{long_group}_{short_group}"

    long_short_daily_ret = (
        group_daily_ret[long_group]
        - group_daily_ret[short_group]
    )

    group_metrics.loc[long_short_name] = performance_metrics(
        long_short_daily_ret,
        benchmark_ret=benchmark_ret,
        annual_rf=RISK_FREE_RATE,
    )

    # ---------- 换手率 ----------
    turnover_metrics = pd.DataFrame({
        "平均日换手率": group_turnover[group_names].mean(),
        "中位数日换手率": group_turnover[group_names].median(),
        "年化换手率": (
            group_turnover[group_names].mean() * TRADING_DAYS
        ),
        "最大单日换手率": group_turnover[group_names].max(),
    })

    total_turnover = group_turnover[group_names].mean(axis=1)

    turnover_metrics.loc[total_group_name] = {
        "平均日换手率": total_turnover.mean(),
        "中位数日换手率": total_turnover.median(),
        "年化换手率": total_turnover.mean() * TRADING_DAYS,
        "最大单日换手率": total_turnover.max(),
    }

    # 多空两腿各 100% 名义本金，对应总换手率为两腿之和
    long_short_turnover = group_turnover[
        [long_group, short_group]
    ].sum(axis=1, min_count=2)

    turnover_metrics.loc[long_short_name] = {
        "平均日换手率": long_short_turnover.mean(),
        "中位数日换手率": long_short_turnover.median(),
        "年化换手率": long_short_turnover.mean() * TRADING_DAYS,
        "最大单日换手率": long_short_turnover.max(),
    }

    # ---------- 写入 TXT ----------
    ic_text = dataframe_to_text(
        ic_metrics.to_frame("Result"),
        scientific_rows={"IC p 值"},
        integer_rows={"样本天数"},
    )

    group_text = dataframe_to_text(group_metrics)
    turnover_text = dataframe_to_text(turnover_metrics)

    summary_text = (
        ic_text
        + "\n"
        + (
            "本次绩效计算使用的基准区间： "
            f"{benchmark_ret.index.min():%Y-%m-%d} 至 "
            f"{benchmark_ret.index.max():%Y-%m-%d}\n"
        )
        + f"策略与基准共同样本天数： {len(benchmark_ret)}\n\n"
        + group_text
        + "\n"
        + turnover_text
    )

    summary_file = save_dir / "summary.txt"

    summary_file.write_text(
        summary_text,
        encoding="utf-8",
    )

    return summary_file


# ========= 批量生成每个因子的 summary.txt =========
result_dirs = sorted({
    ic_file.parent
    for ic_file in data_dir.rglob("ic.csv")
    if not (
        set(ic_file.relative_to(data_dir).parts)
        & ignore_folder_names
    )
})

success_count = 0
failed_count = 0

for save_dir in result_dirs:
    try:
        summary_file = build_summary(save_dir)
        print(f"已生成：{summary_file}")
        success_count += 1

    except Exception as exc:
        print(f"跳过：{save_dir.name}")
        print(f"原因：{exc}")
        failed_count += 1

print(
    f"\n完成：成功 {success_count} 个，失败 {failed_count} 个。"
)

已生成：D:\Download\factor_results\CSAD 模型（基于“小型”板块）\summary.txt
已生成：D:\Download\factor_results\CSAD 模型（基于时序回归变形）\summary.txt
已生成：D:\Download\factor_results\CSAD 模型（基于板块中地位）\summary.txt
已生成：D:\Download\factor_results\LSV 模型\summary.txt
已生成：D:\Download\factor_results\SLSV 模型（对称 LSV 模型）\summary.txt
已生成：D:\Download\factor_results\前景价值（TK）\summary.txt
已生成：D:\Download\factor_results\基于 PB 的前景价值 TK\summary.txt
已生成：D:\Download\factor_results\基于 PE 的分析师锚定效应偏差（CAF_EP）\summary.txt
已生成：D:\Download\factor_results\基于市场波动率的注意力捕捉\summary.txt
已生成：D:\Download\factor_results\基于异常收益的注意力捕捉\summary.txt
已生成：D:\Download\factor_results\基于异常收益的注意力溢出（SPILL）\summary.txt
已生成：D:\Download\factor_results\基于换手率的注意力捕捉\summary.txt
已生成：D:\Download\factor_results\基于换手率的注意力溢出（SPILL）\summary.txt
已生成：D:\Download\factor_results\基于沪深总成交额的注意力捕捉\summary.txt
已生成：D:\Download\factor_results\基于涨跌停的注意力捕捉\summary.txt
已生成：D:\Download\factor_results\平均异常日收益率（ABNRETAVG）\summary.txt
已生成：D:\Download\factor_results\平均换手率（TURNAVG）\summary.txt
已

In [16]:
os.listdir(r"D:\Download\factor_results\CSAD 模型（基于“小型”板块）")

['ic.csv',
 'ic_decay.csv',
 'ic_industry_distribute.csv',
 'ic_summary.csv',
 'params.json',
 'quantile_factor_returns.csv',
 'quantile_turnover.csv',
 'small_board_csad_all_all_daily_none_q10.html',
 'small_board_csad_all_all_daily_none_q10_ic.png',
 'small_board_csad_all_a_df.csv',
 'summary.txt']

---

- 算术年化：平均每天大约赚多少，乘以 252。
- 几何年化：真实复利后，净值每年实际增长多少。

例如连续两天收益：

| 日期 | 日收益 |
|---|---:|
| 第一天 | +10% |
| 第二天 | -10% |

算术平均日收益是：

$$
(10\% - 10\%) / 2 = 0
$$

看起来没有收益。

但实际净值是：

$$
1.10 \times 0.90 = 0.99
$$

两天实际亏损 1%。这就是波动拖累，也只有几何收益能反映出来。

对你的绩效指标：

- `年化收益率` 应使用几何年化，因为它描述资金曲线从起点到终点的真实复利增长。
- `年化收益差（算术）` 是“策略年化收益率 − 基准年化收益率”，便于直观比较两个组合的最终表现。
- `年化超额收益（几何）` 是策略净值相对基准净值的复利增长率：

$$
\left[
\prod_t \frac{1+r_{strategy,t}}{1+r_{benchmark,t}}
\right]^{252/T} - 1
$$

它衡量“持有策略相对于持有基准，财富每年多增长多少”。

- `Alpha（年化）` 通常采用算术年化：

$$
\alpha_{\text{annual}} \approx 252 \times \alpha_{\text{daily}}
$$

因为 Alpha 来自日收益回归的截距，是条件期望收益的线性估计量，不是一个可直接复利的单一投资组合净值。

因此，建议你的表格采用：

- 年化收益率：几何年化
- 年化超额收益（几何）：策略相对基准的真实复利优势
- 年化收益差（算术）：两个年化收益率直接相减
- Alpha（年化）：日 Alpha × 252

这样每个指标的财务含义明确，不会把“期望收益”“实际净值增长”和“回归截距”混为一谈。